In [ ]:
from torch.utils.data import Dataset
import tqdm
import rasterio
from rasterio.mask import mask
import numpy as np
import geopandas as gpd
import torch
import torch.nn.functional as F
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import hashlib
from pathlib import Path
from sklearn.metrics import roc_auc_score

def cache_key(row):
    h = hashlib.md5(row.image_path.encode()).hexdigest()[:8]
    return f"{row.name}_{h}.pt"

SELECTED_BANDS = [0, 12, 13, 14]  # 1,13,14,15 → 0-based
TARGET_SIZE = 32
DEBUG_IMG_DIR = "/app/data/datasets/debug/bush/debug" 
CACHE_DIR = "/app/data/datasets/debug/bush/cache"
MODEL_SAVE_PATH = "/app/data/datasets/debug/bush/models"
gdf_datas = gpd.read_file('/app/data/datasets/debug/bush/target_dol_zones_v1.gpkg',driver='GPKG')
gdf_datas.head(5)

 

In [ ]:

def extract_patch(src_path, geometry, sample_id=None, debug=False):
    """
    Returns tensor shape (4, TARGET_SIZE, TARGET_SIZE)
    """
    with rasterio.open(src_path) as src:  
        #print(f"processing raster image : {src_path} - crs : {src.crs}")
        #print(f"geometry : {geometry}")
        try: 
            out_image, out_transform = mask(
                src,
                [geometry],
                crop=True,
                filled=True,
                nodata=0
            )
        except:
            print(f"ERROR : empty intersection on {src_path}")
            return None

        # select bands
                # -----------------------------
        # Build RGB-like patch
        # -----------------------------
        # Extract bands
        b0 = out_image[0]        # band 0 → R
        b13 = out_image[13]      # band 13
        b14 = out_image[14]      # band 14
        b15 = out_image[15]      # band 15 → B

        # Combine for G
        g = b13 + b14

        # Stack channels: R,G,B
        patch = np.stack([b0, g, b15], axis=0)  # shape (3, H, W)

        if debug:
            pre_path = DEBUG_IMG_DIR + f"/{sample_id}_pre_resize.tif"

            profile = src.profile.copy()
            profile.update(
                {
                    "height": patch.shape[1],
                    "width": patch.shape[2],
                    "transform": out_transform,
                    "count": patch.shape[0],
                    "nodata": 0,
                }
            )

            with rasterio.open(pre_path, "w", **profile) as dst:
                dst.write(patch)

        # normalize per band (robust)
        patch = np.clip(patch, 0, np.percentile(patch, 99))
        patch = patch / (patch.max(axis=(1,2), keepdims=True) + 1e-6)
        
        # to torch
        patch = torch.tensor(patch, dtype=torch.float32)

        # resize
        patch = F.interpolate(
            patch.unsqueeze(0),
            size=(TARGET_SIZE, TARGET_SIZE),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        if debug:
            post_path = DEBUG_IMG_DIR + f"/{sample_id}_post_resize.tif"

            patch_np = patch.cpu().numpy()

            profile = {
                "driver": "GTiff",
                "height": TARGET_SIZE,
                "width": TARGET_SIZE,
                "count": patch_np.shape[0],
                "dtype": patch_np.dtype,
            }

            with rasterio.open(post_path, "w", **profile) as dst:
                dst.write(patch_np)



        return patch

def find_best_threshold(y_true, y_proba, metric="f1", n_steps=100):
    """
    Find best threshold for binary predictions based on a metric.

    Parameters
    ----------
    y_true : np.ndarray or list
        True binary labels (0/1)
    y_proba : np.ndarray or list
        Predicted probabilities for positive class
    metric : str
        "f1" or "accuracy"
    n_steps : int
        Number of thresholds to test between 0 and 1

    Returns
    -------
    best_threshold : float
    best_score : float
    """
    thresholds = np.linspace(0, 1, n_steps)
    best_score = -np.inf
    best_threshold = 0.5  # default fallback

    y_true = np.array(y_true)
    y_proba = np.array(y_proba)

    for t in thresholds:
        y_pred = (y_proba > t).astype(int)

        if metric == "f1":
            score = f1_score(y_true, y_pred)
        elif metric == "accuracy":
            score = accuracy_score(y_true, y_pred)
        else:
            raise ValueError(f"Unknown metric: {metric}")

        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

class GeoRasterDataset(Dataset):
    def __init__(self, gdf, raster_crs):
        
        self.gdf = gdf.reset_index(drop=True)
        self.gdf.to_crs(raster_crs,inplace=True)
        print(f"Dataset geometries set to crs : {raster_crs}")

    def __len__(self):
        return len(self.gdf)

    def __getitem__(self, idx):
        try:
            row = self.gdf.iloc[idx]
            x = extract_patch(row.image_path, row.geometry, sample_id=idx)
            y = torch.tensor(row.target_control, dtype=torch.long)
            
        except Exception as e:
            # 🔥 CRITICAL: raise IndexError to let DataLoader skip
            raise IndexError(f"Invalid sample at index {idx}: {e}") 
            
        return x, y

class GeoRasterCachedDataset(Dataset):
    def __init__(self, gdf, raster_crs, cache_dir):
        self.gdf = gdf.reset_index(drop=True)
        self.gdf.to_crs(raster_crs, inplace=True)
        self.cache_dir = Path(cache_dir)

    def __len__(self):
        return len(self.gdf)

    def __getitem__(self, idx):
        row = self.gdf.iloc[idx]
        key = cache_key(row)
        cache_file = self.cache_dir / key

        if cache_file.exists():
            x = torch.load(cache_file)
        else:
            x = extract_patch(row.image_path, row.geometry, sample_id=idx)
            if x is None:
                raise IndexError(f"Invalid raster patch at index {idx}")
            torch.save(x, cache_file)

        y = torch.tensor(row.target_control, dtype=torch.long)
        return x, y

class ScratchDolConv(nn.Module):
    def __init__(self,save_path):
        super().__init__()
        self.best_acc = 0.0
        self.save_path = save_path
        self.device = "cpu" #torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
        
        self.features = nn.Sequential(
            nn.Conv2d(4, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 2)
        )
        
        self.optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)
    
    def train_epoch(self, loader):
        self.train()
        total_loss = 0
        for x, y in tqdm.tqdm(loader):
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            loss = self.criterion(self(x), y)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)
    
    
    def eval_epoch(self, loader, save_best=True):
        self.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                preds = self(x).argmax(1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        acc = correct / total     
        # 🔥 Save best model
        if save_best and acc > self.best_acc:
            self.best_acc = acc
            torch.save(self.state_dict(), self.save_path)
            print(f"✅ New best model saved (acc={acc:.3f})")
            
        return acc
    
    def predict(self, loader, return_proba=False):
        """
        Parameters
        ----------
        loader : DataLoader
        return_proba : bool
            If True, also return class probabilities

        Returns
        -------
        preds : np.ndarray
            Predicted class labels
        probs : np.ndarray (optional)
            Predicted probabilities
        """
        self.eval()

        all_preds = []
        all_probs = []

        with torch.no_grad():
            for x, _ in loader:
                x = x.to(self.device)
                logits = self(x)

                probs = torch.softmax(logits, dim=1)
                preds = probs.argmax(dim=1)

                all_preds.append(preds.cpu())
                if return_proba:
                    all_probs.append(probs.cpu())

        all_preds = torch.cat(all_preds).numpy()

        if return_proba:
            all_probs = torch.cat(all_probs).numpy()
            return all_preds, all_probs

        return all_preds

    def predict_sample(self, x, return_proba=False):
        """
        x : torch.Tensor or np.ndarray
            Shape (4, H, W) or (1, 4, H, W)

        Returns
        -------
        pred : int
        proba : np.ndarray (optional)
        """
        self.eval()

        # Convert numpy → torch
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)

        # Add batch dim if needed
        if x.dim() == 3:
            x = x.unsqueeze(0)

        x = x.to(self.device)

        with torch.no_grad():
            logits = self(x)
            probs = torch.softmax(logits, dim=1)
            pred = probs.argmax(dim=1).item()

        if return_proba:
            return pred, probs.cpu().numpy()[0]

        return pred
    
class DolConv(nn.Module):
    def __init__(self):
        super().__init__()

        self.best_acc = 0.0
        self.best_auc = 0.5
        self.device = "cuda"  # or cuda if you want
        self.to(self.device)

        # -----------------------------
        # Load pretrained ResNet18
        # -----------------------------
        self.backbone = models.resnet18(weights='ResNet18_Weights.DEFAULT')

        # -----------------------------
        # Adjust first conv for 32x32 images
        # Original: Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        # We'll use smaller kernel + stride 1 + remove first maxpool
        # -----------------------------
        old_conv = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=old_conv.out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=old_conv.bias is not None
        )

        # Remove first maxpool
        self.backbone.maxpool = nn.Identity()

        # -----------------------------
        # Replace classifier head
        # -----------------------------
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 2)
        )


        for name, param in self.backbone.named_parameters():
            if (
                name.startswith("conv1") or
                name.startswith("layer1") or
                name.startswith("layer2")
            ):
                param.requires_grad = False
        # -----------------------------
        # Optim / loss
        # -----------------------------
        self.optimizer = torch.optim.AdamW(
            [
                {"params": self.backbone.layer3.parameters(), "lr": 3e-5},
                {"params": self.backbone.layer4.parameters(), "lr": 1e-4},
                {"params": self.backbone.fc.parameters(), "lr": 1e-3},
            ],
            weight_decay=1e-4
        )
        class_weights = torch.tensor([0.43, 0.57], device=self.device)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)

        self.to(self.device)
        
        self.augment = nn.Sequential(
            # Random horizontal flip
            nn.Identity(),  # placeholder (keeps Sequential happy)
        )

    # --------------------------------------------------
    # Forward
    # --------------------------------------------------
    def forward(self, x):
        if x.dim() != 4 or x.shape[1] != 3:
            raise ValueError(f"Expected input (B,3,H,W), got {x.shape}")
        return self.backbone(x)

    def augment_batch(self, x):
        """
        x : torch.Tensor (B, C, H, W)
        """

        # Random horizontal flip
        if torch.rand(1).item() < 0.5:
            x = torch.flip(x, dims=[3])

        # Random vertical flip
        if torch.rand(1).item() < 0.5:
            x = torch.flip(x, dims=[2])

        # Small random rotation (90° steps only – safe for rasters)
        if torch.rand(1).item() < 0.2:
            x = torch.rot90(x, k=1, dims=[2, 3])

        # Gaussian noise (very mild)
        if torch.rand(1).item() < 0.2:
            noise = torch.randn_like(x) * 0.001
            x = x + noise
            x = torch.clamp(x, 0.0, 1.0)

        return x

    def _save_debug_batch(self, x, prefix, out_dir=DEBUG_IMG_DIR, max_items=2):
        """
        Save a few samples from a batch as simple TIFFs.

        x : torch.Tensor (B, C, H, W)
        prefix : str (e.g. 'pre_aug', 'post_aug')
        """
        from pathlib import Path
        import rasterio

        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        x = x.detach().cpu().numpy()

        for i in range(min(max_items, x.shape[0])):
            path = out_dir / f"{prefix}_sample{i}.tif"

            profile = {
                "driver": "GTiff",
                "height": x.shape[2],
                "width": x.shape[3],
                "count": x.shape[1],
                "dtype": x.dtype,
            }

            with rasterio.open(path, "w", **profile) as dst:
                dst.write(x[i])
    # --------------------------------------------------
    # Training / eval (unchanged)
    # --------------------------------------------------
    def train_epoch(self, loader, debug_aug=True):
        self.train()
        total_loss = 0
        pbar = tqdm.tqdm(loader, desc="Training", leave=False)

        for batch_idx, (x, y) in enumerate(pbar):
            x, y = x.to(self.device), y.to(self.device)
            """
            # -----------------------------
            # DATA AUGMENTATION (train only)
            # -----------------------------
            if debug_aug and batch_idx == 0:
                self._save_debug_batch(
                    x,
                    prefix="pre_aug"
                )
            """ 
            x = self.augment_batch(x)
            """
            if debug_aug and batch_idx == 0:
                self._save_debug_batch(
                    x,
                    prefix="post_aug"
                )
            """
            self.optimizer.zero_grad()
            loss = self.criterion(self(x), y)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": self.optimizer.param_groups[0]["lr"]})
        return total_loss / len(loader)

    def eval_epoch(self, loader, save_best=True, save_path= None):
        self.eval()
        correct, total = 0, 0
        all_probs = []
        all_targets = []
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                logits = self(x)
                probs = torch.softmax(logits, dim=1)

                preds = probs.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
                
                all_probs.append(probs[:, 1].cpu())
                all_targets.append(y.cpu())
            
        acc = correct / total
        all_probs = torch.cat(all_probs).numpy()
        all_targets = torch.cat(all_targets).numpy()

        acc = correct / total

        try:
            auc = roc_auc_score(all_targets, all_probs)
        except ValueError:
            auc = float("nan")  # happens if only one class present

        if save_best and auc > self.best_auc:
            self.best_auc = auc
            torch.save(self.state_dict(), save_path)
            print(f"✅ New best model saved (auc={auc:.3f})")

        return acc, auc

    # --------------------------------------------------
    # Prediction helpers (unchanged)
    # --------------------------------------------------
    def predict(self, loader, return_proba=False):
        self.eval()
        all_preds, all_probs = [], []

        with torch.no_grad():
            for x, _ in loader:
                x = x.to(self.device)
                logits = self(x)
                probs = torch.softmax(logits, dim=1)
                preds = probs.argmax(dim=1)

                all_preds.append(preds.cpu())
                if return_proba:
                    all_probs.append(probs.cpu())

        all_preds = torch.cat(all_preds).numpy()

        if return_proba:
            all_probs = torch.cat(all_probs).numpy()
            return all_preds, all_probs

        return all_preds

    def predict_sample(self, x, return_proba=False):
        self.eval()

        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)

        if x.dim() == 3:
            x = x.unsqueeze(0)

        x = x.to(self.device)

        with torch.no_grad():
            logits = self(x)
            probs = torch.softmax(logits, dim=1)
            pred = probs.argmax(dim=1).item()

        if return_proba:
            return pred, probs.cpu().numpy()[0]

        return pred

In [ ]:
   
test_gdf = gdf_datas[gdf_datas.insee_com=='34035']
train_gdf = gdf_datas[~(gdf_datas.insee_com=='34035')]

train_gdf, val_gdf = train_test_split(
    train_gdf,
    test_size=0.2,
    stratify=train_gdf.target_control,
    random_state=42
)
train_gdf = train_gdf[:int(len(train_gdf)/3*2)]
print(f"trainset size : {len(train_gdf)} - % target control balance : {np.sum(train_gdf.target_control.tolist())/len(train_gdf)}")
print(f"valset size : {len(val_gdf)} - % target control balance : {np.sum(val_gdf.target_control.tolist())/len(val_gdf)}")
print(f"test size : {len(test_gdf)} - % target control balance : {np.sum(test_gdf.target_control.tolist())/len(test_gdf)}")

raster_crs = 'EPSG:2154'

model_save = MODEL_SAVE_PATH + "small_dol_best_v3.pt"

model = DolConv()

train_ds = GeoRasterCachedDataset(train_gdf, raster_crs, CACHE_DIR)
val_ds   = GeoRasterCachedDataset(val_gdf, raster_crs, CACHE_DIR)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4,
    persistent_workers=True,
    pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=16,num_workers=4,
    persistent_workers=True,
    pin_memory=True)

for epoch in range(100):
    loss = model.train_epoch(train_loader)
    acc, auc  = model.eval_epoch(val_loader, save_best=True, save_path=model_save)
    print(f"Epoch {epoch:02d} | loss={loss:.4f} | val_acc={acc:.3f} | val_auc={auc:.3f}")

test_ds   = GeoRasterCachedDataset(test_gdf, raster_crs, CACHE_DIR)

test_loader   = DataLoader(test_ds, batch_size=1,num_workers=1,
    persistent_workers=True,
    pin_memory=True)


model.load_state_dict(torch.load(model_save, map_location=model.device))
model.eval()

y_val_pred = model.predict(val_loader,return_proba=True)
val_gdf = val_gdf.copy()
val_gdf['pred_control'] = y_val_pred[0].tolist()
val_gdf['proba_control'] = y_val_pred[1][:,1].tolist()
y_true = val_gdf['target_control'].values
y_proba = val_gdf['proba_control'].values 
best_thresh, best_score = find_best_threshold(y_true, y_proba, metric="f1")
print(f"Best threshold = {best_thresh:.3f}, F1-score = {best_score:.3f}")

y_pred = model.predict(test_loader,return_proba=True)
test_gdf = test_gdf.copy()
test_gdf['pred_control'] = y_pred[0].tolist()
test_gdf['proba_control'] = y_pred[1][:,1].tolist()
test_gdf['pred_control_best'] = (test_gdf['proba_control'] > best_thresh).astype(int)

test_gdf.to_file('/app/data/datasets/debug/bush/pred_small_dol_zones_vX.gpkg',driver='GPKG')